# v47 — 構造化系列デコード(候補21状態の CRF + Viterbi 経路選択)

「系列的な仮の正解(各PF軌跡)から、系列整合を保ってどの経路を辿るか/どの仮正解を採るか」を選ぶタスク。
soft凸結合は候補の凸包までしか行けない(頭打ち≈6.4)。→ **候補21を離散状態とする線形鎖CRF**にし、
- Encoder(1軸Transformer)= 位置×候補の emission。
- 遷移 = 学習行列 + **連続性コスト ∝ |Δdelta(乗換点)|**(滑らかな経路ほど得)。
- **Viterbi で大域最適経路をハード選択**(凸包の壁を破りoracleへ)。連続性項が飛び移りを抑制。
学習=oracle最良を教師にCRF-NLL + soft周辺MSE補助。推論=Viterbi(ハード)/周辺(ソフト)両方比較。v47 artifactのみ参照。


In [ ]:
# ===== 設定(v47 artifact のみ / 省メモリ) =====
import os, json, time, math, gc
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import GroupKFold
ART=Path(r"C:\Users\kosaka256\Documents\rogii_claude")/"v47"/"artifacts_v47"
NPZ=ART/"candidates_v47.npz"; META=json.loads((ART/"candidates_v47_meta.json").read_text(encoding="utf-8"))
assert NPZ.exists(),"先に python v47/build_candidates_v47.py"
STRIDE=16; T_MAX=int(os.environ.get("V47_TMAX","256")); SCALE,SG=30.0,20.0; N_SPLITS=5
D_MODEL,N_LAYER,N_HEAD,D_FF,DROP=128,3,8,256,0.1
LAM_SOFT=0.3            # soft周辺MSEの補助重み
EPOCHS=int(os.environ.get("V47_EPOCHS","40")); BATCH=int(os.environ.get("V47_BATCH","16")); LR=3e-4
N_WELLS_DBG=int(os.environ.get("V47_NWELLS","0")); USE_AMP=torch.cuda.is_available()
SEED=42; DEV="cuda" if torch.cuda.is_available() else "cpu"; torch.manual_seed(SEED); np.random.seed(SEED)
try: torch.cuda.set_per_process_memory_fraction(0.85)
except Exception: pass
CAND_NAMES=META["cand_names"]; NC=len(CAND_NAMES)
print("device",DEV,"| 候補",NC,"| T_MAX",T_MAX,"| AMP",USE_AMP)


In [ ]:
# ===== ロード =====
t0=time.time(); z=np.load(NPZ)
DEL=z["DEL"]; y=z["y"]; lk=z["lk"].astype(np.float64); mds=z["mds"].astype(np.float64); g=z["wellcode"].astype(np.int64); N=len(y)
STD=z["STD"]; CTX=z["ctx"]; GRC=z["GRC"]; GRC2=z["GRC2"]; GRM=z["GRM"]; STRD=z["STRD"]; STRS=z["STRS"]
single={nm:float(np.sqrt(np.mean((DEL[:,j]-y)**2))) for j,nm in enumerate(CAND_NAMES)}
best_single=min(single,key=single.get); jbest=CAND_NAMES.index(best_single)
orc=float(np.sqrt(np.mean(np.min(np.abs(DEL-y[:,None]),1)**2)))
print(f"best単体={best_single}:{single[best_single]:.4f} mean={np.sqrt(np.mean((DEL.mean(1)-y)**2)):.4f} ORACLE={orc:.4f} ({time.time()-t0:.0f}s)")


In [ ]:
# ===== bin化→固定長チャンク + 平坦特徴 =====
order=np.lexsort((mds,g)); gg=g[order]; _,first=np.unique(gg,return_index=True); first=np.sort(first); bounds=list(first)+[N]
wells=[int(gg[bounds[k]]) for k in range(len(bounds)-1)]
if N_WELLS_DBG>0: keep=min(N_WELLS_DBG,len(wells)); wells=wells[:keep]; bounds=bounds[:keep+1]
DELo=DEL[order]; STDo=STD[order]; yo=y[order]; CTXo=CTX[order]; GRCo=GRC[order]; GRC2o=GRC2[order]; GRMo=GRM[order]; STRDo=STRD[order]; STRSo=STRS[order]; pos_g=order.copy()
def bmean(a,bid,nb):
    out=np.zeros((nb,)+a.shape[1:],np.float32); cnt=np.zeros(nb,np.float32); np.add.at(out,bid,a); np.add.at(cnt,bid,1.0)
    return out/np.maximum(cnt.reshape((nb,)+(1,)*(a.ndim-1)),1.0), cnt
def well_arr(s,e):
    n=e-s; nb=(n+STRIDE-1)//STRIDE; bid=np.arange(n)//STRIDE
    d,cnt=bmean(DELo[s:e],bid,nb); st,_=bmean(STDo[s:e],bid,nb); yb,_=bmean(yo[s:e],bid,nb)
    gc1,_=bmean(GRCo[s:e],bid,nb); gc2,_=bmean(GRC2o[s:e],bid,nb); gm,_=bmean(GRMo[s:e],bid,nb)
    strd,_=bmean(STRDo[s:e][:,None],bid,nb); strs,_=bmean(STRSo[s:e][:,None],bid,nb); cx,_=bmean(CTXo[s:e],bid,nb)
    med=np.median(d,1,keepdims=True); ddev=d-med; spread=(np.percentile(d,90,1)-np.percentile(d,10,1))[:,None]
    rank=np.argsort(np.argsort(gc1,1),1).astype(np.float32)/(NC-1); sagree=d-strd
    feat=np.concatenate([d/SCALE, np.log1p(np.maximum(st,0)), gc1/SG, gc2/SG, gm/SG, ddev/SCALE, rank, sagree/SCALE,
                         strd/SCALE, np.log1p(np.maximum(strs,0)), spread/SCALE,
                         np.log1p(np.maximum(cx[:,0:1],0)), cx[:,1:2], np.log1p(np.maximum(cx[:,2:3],0))],1).astype(np.float32)
    return feat, d.astype(np.float32), yb.astype(np.float32), cnt.astype(np.float32), bid, gc1.astype(np.float32)
SEQ=[]; CH_WELL=[]
for k in range(len(wells)):
    s,e=bounds[k],bounds[k+1]; feat,d,yb,cnt,bid,gc1=well_arr(s,e); nb=len(d); pos=pos_g[s:e]
    for c0 in range(0,nb,T_MAX):
        c1=min(c0+T_MAX,nb); rows=np.where((bid>=c0)&(bid<c1))[0]
        SEQ.append(dict(feat=feat[c0:c1],dbin=d[c0:c1],ybin=yb[c0:c1],cnt=cnt[c0:c1],gc1=gc1[c0:c1],bid=(bid[rows]-c0),pos=pos[rows]))
        CH_WELL.append(wells[k])
FC=SEQ[0]["feat"].shape[1]; CH_WELL=np.array(CH_WELL)
del DELo,STDo,yo,CTXo,GRCo,GRC2o,GRMo,STRDo,STRSo,STD,CTX,GRC,GRC2,GRM,STRS; gc.collect()
print(f"チャンク数 {len(SEQ)} | 特徴次元 {FC} | 最大bin {max(len(s['dbin']) for s in SEQ)}")


In [ ]:
# ===== Encoder(1軸Transformer) + 線形鎖CRF(連続性遷移) =====
class PosEnc(nn.Module):
    def __init__(s,d,maxlen=2048):
        super().__init__(); pe=torch.zeros(maxlen,d); p=torch.arange(maxlen).unsqueeze(1).float()
        dv=torch.exp(torch.arange(0,d,2).float()*(-math.log(10000.0)/d)); pe[:,0::2]=torch.sin(p*dv); pe[:,1::2]=torch.cos(p*dv); s.register_buffer("pe",pe)
    def forward(s,T): return s.pe[:T]
class Encoder(nn.Module):
    def __init__(s,fc,d=D_MODEL,layers=N_LAYER,h=N_HEAD,ff=D_FF,drop=DROP,ncand=NC):
        super().__init__(); s.proj=nn.Sequential(nn.Linear(fc,d),nn.GELU(),nn.LayerNorm(d)); s.pe=PosEnc(d)
        enc=nn.TransformerEncoderLayer(d,h,ff,drop,batch_first=True,norm_first=True); s.enc=nn.TransformerEncoder(enc,layers)
        s.emit=nn.Linear(d,ncand)
    def forward(s,feat,pad):
        x=s.proj(feat)+s.pe(feat.size(1)).to(feat.device)[None]; x=s.enc(x,src_key_padding_mask=pad); return s.emit(x)  # [B,T,NC]

class ChainCRF(nn.Module):
    """位置依存の連続性遷移つき線形鎖CRF。状態=21候補。"""
    def __init__(s,ncand=NC):
        super().__init__(); s.trans=nn.Parameter(torch.zeros(ncand,ncand)); s.g=nn.Parameter(torch.tensor(0.0))  # softplus->gamma
        s.start=nn.Parameter(torch.zeros(ncand)); s.end=nn.Parameter(torch.zeros(ncand)); s.nc=ncand
    def _trT(s,dbin,t):        # 遷移スコア[B,i(prev),j(cur)] at step t
        gamma=F.softplus(s.g)
        cont=(dbin[:,t][:,None,:]-dbin[:,t-1][:,:,None]).abs()   # |delta_j(t)-delta_i(t-1)| [B,i,j]? note broadcasting
        return s.trans[None]-gamma*cont/30.0
    def nll(s,emis,dbin,mask,tags):    # emis[B,T,NC] mask[B,T](1=有効) tags[B,T]long
        B,T,C=emis.shape
        a=s.start[None]+emis[:,0]                              # [B,C]
        sc=s.start[tags[:,0]]+emis[torch.arange(B),0,tags[:,0]]
        for t in range(1,T):
            tr=s._trT(dbin,t)                                  # [B,C,C] i=prev,j=cur
            na=emis[:,t]+torch.logsumexp(a[:,:,None]+tr,dim=1) # [B,C]
            mt=mask[:,t][:,None]; a=torch.where(mt.bool(),na,a)
            trg=tr[torch.arange(B),tags[:,t-1],tags[:,t]]
            step=emis[torch.arange(B),t,tags[:,t]]+trg
            sc=sc+torch.where(mask[:,t].bool(),step,torch.zeros_like(step))
        logZ=torch.logsumexp(a+s.end[None],dim=1)
        sc=sc+s.end[tags[torch.arange(B),(mask.sum(1)-1).long()]]
        return (logZ-sc).mean()
    def marginals(s,emis,dbin,mask):   # forward-backward → p[B,T,C]
        B,T,C=emis.shape
        al=[None]*T; al[0]=s.start[None]+emis[:,0]
        for t in range(1,T):
            tr=s._trT(dbin,t); na=emis[:,t]+torch.logsumexp(al[t-1][:,:,None]+tr,1)
            al[t]=torch.where(mask[:,t][:,None].bool(),na,al[t-1])
        be=[None]*T; be[T-1]=s.end[None].expand(B,C).clone()
        for t in range(T-2,-1,-1):
            tr=s._trT(dbin,t+1); nb=torch.logsumexp(tr+emis[:,t+1][:,None,:]+be[t+1][:,None,:],2)
            be[t]=torch.where(mask[:,t+1][:,None].bool(),nb,be[t+1])
        A=torch.stack(al,1); Bt=torch.stack(be,1); p=torch.softmax(A+Bt,dim=2); return p
    def viterbi(s,emis,dbin,mask):
        B,T,C=emis.shape; sc=s.start[None]+emis[:,0]; bps=[]
        ident=torch.arange(C,device=emis.device)[None].expand(B,C)
        for t in range(1,T):
            tr=s._trT(dbin,t); best,arg=(sc[:,:,None]+tr).max(1)               # over i(prev) -> [B,j]
            upd=mask[:,t][:,None].bool(); sc=torch.where(upd,emis[:,t]+best,sc)
            bps.append(torch.where(upd,arg,ident))                            # pad=identity
        bps=torch.stack(bps,1) if bps else torch.zeros(B,0,C,dtype=torch.long,device=emis.device)  # [B,T-1,C]
        last=(mask.sum(1)-1).long(); path=torch.zeros(B,T,dtype=torch.long,device=emis.device)
        cur=(sc+s.end[None]).argmax(1)                                        # 最終有効位置の状態
        for b in range(B):
            L=int(last[b].item()); st=int(cur[b].item()); path[b,L]=st
            for t in range(L,0,-1): st=int(bps[b,t-1,st].item()); path[b,t-1]=st
        return path
print("modules ready. Encoder params ~", round(sum(p.numel() for p in Encoder(FC).parameters())/1e6,2),"M")


In [ ]:
# ===== 学習(CRF-NLL + soft周辺MSE) → 行レベル OOF(Viterbi/soft) =====
def collate(batch):
    T=max(len(b["dbin"]) for b in batch); B=len(batch)
    feat=torch.zeros(B,T,FC); dbin=torch.zeros(B,T,NC); ybin=torch.zeros(B,T); mask=torch.zeros(B,T)
    for i,b in enumerate(batch):
        t=len(b["dbin"]); feat[i,:t]=torch.from_numpy(b["feat"]); dbin[i,:t]=torch.from_numpy(b["dbin"]); ybin[i,:t]=torch.from_numpy(b["ybin"]); mask[i,:t]=1.0
    return feat,dbin,ybin,mask
LEN=np.array([len(s["dbin"]) for s in SEQ])
def batches(idxs): idxs=sorted(idxs,key=lambda i:LEN[i]); return [idxs[c0:c0+BATCH] for c0 in range(0,len(idxs),BATCH)]
folds=list(GroupKFold(N_SPLITS).split(np.arange(len(SEQ)),groups=CH_WELL)); oof=np.full(N,np.nan,np.float32); oof_soft=np.full(N,np.nan,np.float32)
def fill_oof(enc,crf,idxs):
    enc.eval(); crf.eval()
    with torch.no_grad():
        for bt in batches(idxs):
            sub=[SEQ[i] for i in bt]; feat,dbin,ybin,mask=[t.to(DEV) for t in collate(sub)]
            with torch.autocast("cuda",enabled=USE_AMP): emis=enc(feat,mask==0)
            emis=emis.float(); path=crf.viterbi(emis,dbin,mask); p=crf.marginals(emis,dbin,mask)
            path=path.cpu().numpy(); soft=(p*dbin).sum(-1).cpu().numpy()
            for bi,b in enumerate(sub):
                sel=path[bi][b["bid"]]; oof[b["pos"]]=DEL[b["pos"], sel].astype(np.float32)
                oof_soft[b["pos"]]=soft[bi][b["bid"]].astype(np.float32)
t0=time.time()
for k,(tr,va) in enumerate(folds):
    enc=Encoder(FC).to(DEV); crf=ChainCRF().to(DEV)
    opt=torch.optim.AdamW(list(enc.parameters())+list(crf.parameters()),lr=LR,weight_decay=1e-4)
    scaler=torch.cuda.amp.GradScaler(enabled=USE_AMP); trb=batches(list(tr)); sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS*max(1,len(trb)))
    best=1e9; pat=0; fpe=ART/f"crf_enc{k}.pt"; fpc=ART/f"crf_crf{k}.pt"
    for ep in range(EPOCHS):
        enc.train(); crf.train(); np.random.shuffle(trb); tl=0.0; nb=0
        for bt in trb:
            sub=[SEQ[i] for i in bt]; feat,dbin,ybin,mask=[t.to(DEV) for t in collate(sub)]
            gold=(dbin-ybin.unsqueeze(-1)).abs().argmin(-1)
            with torch.autocast("cuda",enabled=USE_AMP): emis=enc(feat,mask==0)
            emis=emis.float(); nll=crf.nll(emis,dbin,mask,gold)
            p=crf.marginals(emis,dbin,mask); soft=(p*dbin).sum(-1); mse=(((soft-ybin)**2)*mask).sum()/mask.sum().clamp_min(1.0)
            loss=nll+LAM_SOFT*mse
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(list(enc.parameters())+list(crf.parameters()),1.0); scaler.step(opt); scaler.update(); sch.step()
            tl+=float(mse); nb+=1
        vs=0.0; vn=0; enc.eval(); crf.eval()
        with torch.no_grad():
            for bt in batches(list(va)):
                sub=[SEQ[i] for i in bt]; feat,dbin,ybin,mask=[t.to(DEV) for t in collate(sub)]
                with torch.autocast("cuda",enabled=USE_AMP): emis=enc(feat,mask==0)
                emis=emis.float(); p=crf.marginals(emis,dbin,mask); soft=(p*dbin).sum(-1); vs+=float((((soft-ybin)**2)*mask).sum()/mask.sum().clamp_min(1.0))*len(sub); vn+=len(sub)
        vrm=math.sqrt(vs/max(vn,1))
        if vrm<best-1e-4: best=vrm; pat=0; torch.save(enc.state_dict(),fpe); torch.save(crf.state_dict(),fpc)
        else: pat+=1
        if ep%5==0 or ep==EPOCHS-1: print(f" fold{k} ep{ep} soft_train={math.sqrt(tl/max(nb,1)):.4f} soft_val={vrm:.4f}{' *' if pat==0 else ''} ({time.time()-t0:.0f}s)",flush=True)
        if pat>=8: print(f" fold{k} early stop ep{ep}"); break
    enc.load_state_dict(torch.load(fpe)); crf.load_state_dict(torch.load(fpc)); fill_oof(enc,crf,list(va))
    print(f"== fold{k} OOF埋め {np.mean(np.isfinite(oof))*100:.1f}%")
    del enc,crf,opt,scaler; gc.collect();
    if DEV=="cuda": torch.cuda.empty_cache()
np.save(ART/"oof_crf_viterbi.npy",oof); np.save(ART/"oof_crf_soft.npy",oof_soft); print("OOF saved.")


In [ ]:
# ===== 評価(行レベル overall RMSE): Viterbi(ハード) vs soft周辺 =====
from scipy.signal import savgol_filter
m=np.isfinite(oof)
def rmse(a): e=(a-y)[m]; return float(np.sqrt(np.mean(e*e)))
def sg(delta):
    tvt=lk+delta; out=tvt.copy(); oidx=np.argsort(g,kind="stable"); gs=g[oidx]; bnd=np.flatnonzero(np.diff(gs))+1
    for rows in np.split(oidx,bnd):
        rr=rows[m[rows]];
        if len(rr)<5: continue
        rr=rr[np.argsort(mds[rr])]; v=tvt[rr]; wl=min(17,len(v)); wl-=(wl%2==0)
        if wl>=5: out[rr]=savgol_filter(v,wl,3)
    return float(np.sqrt(np.mean(((out-(lk+y))[m])**2)))
orc=rmse(np.take_along_axis(DEL,np.argmin(np.abs(DEL-y[:,None]),1,keepdims=True),1)[:,0])
print("=== v47 CRF 系列デコード OOF 行レベル overall RMSE ===")
print(f"  Viterbi(ハード経路)  raw={rmse(oof):.4f}  +SG={sg(oof):.4f}")
print(f"  soft(周辺分布凸結合)  raw={rmse(oof_soft):.4f}  +SG={sg(oof_soft):.4f}")
print(f"  参考 best単体={single[best_single]:.4f}  mean={rmse(DEL.mean(1))if False else np.sqrt(np.mean((DEL.mean(1)-y)[m]**2)):.4f}  ORACLE={orc:.4f}")
